# Exercise 3: Binary segmentation with a small U-Net

Synthetic slices: a bright ellipse (the organ, label 1) and a bright square distractor (label 0) on noise; about 20% of slices have no organ. A small U-Net is trained with BCE + soft Dice and measured with Dice per slice.

**Interviewer:** "This notebook doesn't work, and once it runs the results look strange. Walk me through it, tell me what you're checking, and fix it."

There are **10 problems**: 3 crash the code, 7 are silent (the code runs, but the result is wrong).

**How to work** (same as the real interview)
1. Read the whole notebook first and say out loud what each part does.
2. Run the cells from the top and fix the crashes one at a time.
3. Once it runs, go through it again for the silent bugs: check shapes, dtypes, value ranges, split sizes, train vs eval mode, and whether the numbers make sense.

*Level: Role level (segmentation), about 40 minutes. In Colab: Runtime → Change runtime type → T4 GPU (it also runs on a CPU, just slower).*

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset

SEED = 0
SIZE = 64
BATCH_SIZE = 16
EPOCHS = 10
LR = 1e-3

torch.manual_seed(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", device)

### Data

In [ ]:
def make_dataset(n, seed):
    """Draws n slices at 96 x 96. Returns images (n, 96, 96) float32 and masks (n, 96, 96) float32 in {0, 1}."""
    rng = np.random.default_rng(seed)
    big = 96
    yy, xx = np.mgrid[:big, :big]
    imgs = np.zeros((n, big, big), np.float32)
    masks = np.zeros((n, big, big), np.float32)
    for i in range(n):
        img = rng.normal(0.0, 0.3, (big, big))
        if rng.random() > 0.2:
            cy, cx = rng.uniform(25, 71, 2)
            ry, rx = rng.uniform(8, 20, 2)
            organ = ((yy - cy) / ry) ** 2 + ((xx - cx) / rx) ** 2 <= 1
            img[organ] += 1.0
            masks[i] = organ
        sy, sx = rng.integers(5, 80, 2)
        img[sy:sy + 12, sx:sx + 12] += 1.0
        imgs[i] = img
    return imgs, masks


def resize(imgs, masks, size=SIZE):
    x = torch.from_numpy(imgs).unsqueeze(1)                       # (n, 1, 96, 96)
    m = torch.from_numpy(masks).unsqueeze(1)
    x = F.interpolate(x, size=(size, size), mode="bilinear", align_corners=False)
    m = F.interpolate(m, size=(size, size), mode="bilinear", align_corners=False)
    return x.squeeze(1).numpy(), m.squeeze(1).numpy()


class SegDataset(Dataset):
    def __init__(self, imgs, masks, augment=False):
        self.imgs = torch.from_numpy(imgs).unsqueeze(1)           # (n, 1, H, W)
        self.masks = torch.from_numpy(masks)
        self.augment = augment

    def __len__(self):
        return len(self.imgs)

    def __getitem__(self, i):
        img, mask = self.imgs[i], self.masks[i]
        if self.augment:
            if torch.rand(1).item() < 0.5:
                img, mask = img.flip(-1), mask.flip(-1)
            k = int(torch.randint(0, 4, (1,)))
            img = torch.rot90(img, k, (-2, -1))
        return img, mask


train_ds = SegDataset(*resize(*make_dataset(800, seed=1)), augment=True)
val_ds = SegDataset(*resize(*make_dataset(100, seed=1)))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=50)

x, m = next(iter(train_loader))
print("images:", tuple(x.shape), x.dtype, "| masks:", tuple(m.shape), m.dtype, "| mask values:", torch.unique(m).tolist())
print("organ pixels in train masks:", f"{train_ds.masks.mean().item():.3f}")

### Model

In [ ]:
def conv_block(cin, cout):
    return nn.Sequential(
        nn.Conv2d(cin, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(),
        nn.Conv2d(cout, cout, 3, padding=1), nn.BatchNorm2d(cout), nn.ReLU(),
    )


class TinyUNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.enc1 = conv_block(1, 16)                                   # 64 x 64
        self.enc2 = conv_block(16, 32)                                  # 32 x 32
        self.bottleneck = conv_block(32, 64)                            # 16 x 16
        self.pool = nn.MaxPool2d(2)
        self.up2 = nn.ConvTranspose2d(64, 32, kernel_size=2, stride=2)  # 16 -> 32
        self.dec2 = conv_block(64, 32)                                  # 32 (upsampled) + 32 (skip) channels in
        self.up1 = nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2)
        self.dec1 = conv_block(32, 16)
        self.head = nn.Conv2d(16, 1, kernel_size=1)

    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        b = self.bottleneck(self.pool(e2))
        d2 = self.dec2(torch.cat([self.up2(b), e2], dim=0))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(d1)                                            # (B, 1, H, W) raw logits


model = TinyUNet().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

### Loss and metric

In [ ]:
bce = nn.BCEWithLogitsLoss()


def soft_dice_loss(logits, target, eps=1.0):
    probs = logits
    inter = (probs * target).sum(dim=(1, 2, 3))
    total = probs.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    return 1 - ((2 * inter + eps) / (total + eps)).mean()


def loss_fn(logits, target):
    return bce(torch.sigmoid(logits), target) + soft_dice_loss(logits, target)


@torch.no_grad()
def dice_per_slice(logits, target, eps=1e-6):
    pred = (torch.sigmoid(logits) > 0).float()
    inter = (pred * target).sum(dim=(1, 2, 3))
    total = pred.sum(dim=(1, 2, 3)) + target.sum(dim=(1, 2, 3))
    return (2 * inter) / total

### Training

In [ ]:
has_organ = val_ds.masks.sum(dim=(1, 2, 3)) > 0
for epoch in range(1, EPOCHS + 1):
    model.train()
    losses = []
    for x, m in train_loader:
        x, m = x.to(device), m.to(device)
        optimizer.zero_grad()
        loss = loss_fn(model(x), m)
        loss.backward()
        optimizer.step()
        losses.append(loss.item())

    model.eval()
    with torch.no_grad():
        dice = torch.cat([dice_per_slice(model(x.to(device)), m.to(device)).cpu() for x, m in val_loader])
    print(f"epoch {epoch}: train loss {np.mean(losses):.3f} | val Dice all {dice.mean():.3f}"
          f" | with organ {dice[has_organ].mean():.3f} | empty slices {dice[~has_organ].mean():.3f}")

### Your notes
Write each bug like this:
> Bug: cell … → why it's wrong → my fix → crash or silent?